<a href="https://colab.research.google.com/github/banteamlak1888/Fundamentals-for-AI/blob/main/Project_1_CCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Customer Churn Prediction - Group-by-Group Modular Implementation
Each group executes its specific phase and prints its corresponding output.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

def run_group_1_data_show():
    print("=" * 60)
    print("GROUP 1: DATA SHOW & INGESTION")
    print("=" * 60)

    train_path = "customer_churn_dataset-training-master.csv"
    test_path = "customer_churn_dataset-testing-master.csv"

    if not os.path.exists(train_path) or not os.path.exists(test_path):
        print("[INFO] Dataset files not found in root directory. Creating a synthetic sample for demonstration...")
        # Create dummy dataset for demonstration if files don't exist
        np.random.seed(42)
        n_samples = 500
        df_dummy = pd.DataFrame({
            'CustomerID': range(1000, 1000 + n_samples),
            'Age': np.random.randint(18, 70, size=n_samples),
            'Tenure': np.random.randint(1, 60, size=n_samples),
            'MonthlyCharges': np.random.uniform(20.0, 120.0, size=n_samples),
            'Contract': np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], size=n_samples),
            'Churn': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2])
        })
        df_dummy.to_csv(train_path, index=False)
        df_dummy.to_csv(test_path, index=False)

    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)

    print("\n--- Training Dataset Preview (First 5 Rows) ---")
    print(df_train.head())
    print("\nTraining Dataset Shape:", df_train.shape)
    print("Testing Dataset Shape:", df_test.shape)
    print("\nDataset Info:")
    print(df_train.info())
    return df_train, df_test



In [2]:
def run_group_2_exploratory_batch(df_train):
    print("\n" + "=" * 60)
    print("GROUP 2: EXPLORATORY BATCH (EDA & STATISTICS)")
    print("=" * 60)

    target_col = 'Churn' if 'Churn' in df_train.columns else df_train.columns[-1]

    print("\n--- Missing Values Check ---")
    print(df_train.isnull().sum())

    print("\n--- Target Class Distribution ---")
    class_counts = df_train[target_col].value_counts()
    class_props = df_train[target_col].value_counts(normalize=True) * 100
    for cls in class_counts.index:
        print(f"Class {cls}: {class_counts[cls]} samples ({class_props[cls]:.2f}%)")

    print("\n--- Numerical Summary Statistics ---")
    print(df_train.describe())



In [8]:
def run_group_3_train_test_prep(df_train, df_test):
    print("\n" + "=" * 60)
    print("GROUP 3: TRAIN AND TEST PREPARATION")
    print("=" * 60)

    df_train.columns = df_train.columns.str.strip()
    df_test.columns = df_test.columns.str.strip()

    target_col = 'Churn' if 'Churn' in df_train.columns else df_train.columns[-1]

    # Drop rows where target variable is NaN
    df_train = df_train.dropna(subset=[target_col])
    if df_test is not None and target_col in df_test.columns:
        df_test = df_test.dropna(subset=[target_col])

    # Drop identifier columns if present
    drop_cols = ['CustomerID', 'customerID', 'id', 'ID']
    X_train = df_train.drop(columns=[target_col] + [c for c in drop_cols if c in df_train.columns], errors='ignore')
    y_train = df_train[target_col]

    X_test = df_test.drop(columns=[target_col] + [c for c in drop_cols if c in df_test.columns], errors='ignore')
    y_test = df_test[target_col] if target_col in df_test.columns else None

    numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    print(f"Identified Target Variable: {target_col}")
    print(f"Identified Numerical Features ({len(numeric_features)}): {numeric_features}")
    print(f"Identified Categorical Features ({len(categorical_features)}): {categorical_features}")
    print(f"Training feature set shape: {X_train.shape}")
    print(f"Testing feature set shape: {X_test.shape}")

    return X_train, y_train, X_test, y_test, numeric_features, categorical_features

In [4]:
def run_group_4_modeling_and_results(X_train, y_train, X_test, y_test, num_feat, cat_feat):
    print("\n" + "=" * 60)
    print("GROUP 4: MODEL TRAINING & RESULTS EVALUATION")
    print("=" * 60)

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, num_feat),
            ('cat', categorical_transformer, cat_feat)
        ])

    # 4.1 Baseline Model (Logistic Regression)
    print("\n--- [4.1] Baseline Model: Logistic Regression ---")
    baseline_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ])

    baseline_pipeline.fit(X_train, y_train)
    y_train_pred = baseline_pipeline.predict(X_train)
    print("Baseline Training Accuracy: {:.4f}".format(accuracy_score(y_train, y_train_pred)))

    # 4.2 Improved Model (Random Forest)
    print("\n--- [4.2] Improved Model: Random Forest Classifier ---")
    improved_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10))
    ])

    improved_pipeline.fit(X_train, y_train)

    # Cross-Validation Evaluation Results
    cv_scores = cross_val_score(improved_pipeline, X_train, y_train, cv=5, scoring='f1')
    print(f"5-Fold Cross-Validation F1 Scores: {np.round(cv_scores, 4)}")
    print(f"Mean CV F1 Score: {cv_scores.mean():.4f}")

    # Test Set or Final Results Output
    if y_test is not None and len(y_test) > 0:
        print("\n--- [4.3] Final Test Set Evaluation Results ---")
        y_test_pred = improved_pipeline.predict(X_test)
        print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
        print(f"Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
        print(f"Recall:    {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
        print(f"F1-Score:  {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_test_pred))

    return improved_pipeline



In [9]:
def run_group_5_system_testing(model, X_train):
    print("\n" + "=" * 60)
    print("GROUP 5: SYSTEM & EDGE CASE TESTING")
    print("=" * 60)

    test_cases = [
        ("Standard Record", X_train.iloc[0:1]),
        ("Batch Records (5 rows)", X_train.iloc[0:5])
    ]

    for case_name, sample in test_cases:
        try:
            preds = model.predict(sample)
            print(f"[TEST PASSED] {case_name} -> Predictions: {list(preds)}")
        except Exception as e:
            print(f"[TEST FAILED] {case_name} -> Error: {e}")

def main():
    df_train, df_test = run_group_1_data_show()
    run_group_2_exploratory_batch(df_train)
    X_train, y_train, X_test, y_test, num_feat, cat_feat = run_group_3_train_test_prep(df_train, df_test)
    trained_model = run_group_4_modeling_and_results(X_train, y_train, X_test, y_test, num_feat, cat_feat)
    run_group_5_system_testing(trained_model, X_train)

    print("\n" + "=" * 60)
    print("ALL GROUPS EXECUTED SUCCESSFULLY")
    print("=" * 60)

if __name__ == "__main__":
    main()

GROUP 1: DATA SHOW & INGESTION

--- Training Dataset Preview (First 5 Rows) ---
   CustomerID   Age  Gender  Tenure  Usage Frequency  Support Calls  \
0         2.0  30.0  Female    39.0             14.0            5.0   
1         3.0  65.0  Female    49.0              1.0           10.0   
2         4.0  55.0  Female    14.0              4.0            6.0   
3         5.0  58.0    Male    38.0             21.0            7.0   
4         6.0  23.0    Male    32.0             20.0            5.0   

   Payment Delay Subscription Type Contract Length  Total Spend  \
0           18.0          Standard          Annual        932.0   
1            8.0             Basic         Monthly        557.0   
2           18.0             Basic       Quarterly        185.0   
3            7.0          Standard         Monthly        396.0   
4            8.0             Basic         Monthly        617.0   

   Last Interaction  Churn  
0              17.0    1.0  
1               6.0    1.0  
2  